# Football analysis on a Colab GPU

Inference is 87% of this pipeline's runtime (318s of 366s for a 20s clip on CPU).
A GPU removes almost all of that, and more importantly makes `imgsz=1280`
affordable — which takes ball detection from about 32% of frames to 98%.

**Before running:** set the runtime to a GPU.
Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.

## 1. Confirm there is actually a GPU

If this says no GPU, stop and change the runtime type — everything below
will silently fall back to CPU and be no faster than your laptop.

In [ ]:
import torch

print('torch', torch.__version__, '| CUDA build:', torch.version.cuda)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'device: {name}  ({total:.1f} GB)')
else:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU.')

## 2. Mount Google Drive

This is the mounting step. It opens an authorisation prompt — pick your
Google account and allow access. Drive then appears at
`/content/drive/MyDrive/`, and you read and write it like any local folder.

The mount lasts for the session only; re-run this cell after a disconnect.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Paths

Expected layout in Drive:

```
MyDrive/football_analysis/
    code.zip          <- the project, no videos (made for you locally)
    best.pt           <- model weights, 165 MB
    videos/
        city_vs_spurs.mp4
```

Upload once and it stays there; only the code zip changes as we iterate.

In [ ]:
import os

DRIVE = '/content/drive/MyDrive/football_analysis'
VIDEO_NAME = 'city_vs_spurs.mp4'
START, END = '14', '34'
IMGSZ = 1280          # 640 to match the local runs, 1280 to find the ball
BATCH = 32            # raise if the GPU has memory to spare

WORK = '/content/football'

for path in (DRIVE, f'{DRIVE}/code.zip', f'{DRIVE}/best.pt',
             f'{DRIVE}/videos/{VIDEO_NAME}'):
    print(('found  ' if os.path.exists(path) else 'MISSING'), path)

## 4. Dependencies

Colab already ships a CUDA build of torch, so do **not** let anything
reinstall it — that is the usual way a Colab GPU session ends up on CPU.
The check afterwards confirms torch survived.

In [ ]:
!pip -q install ultralytics supervision

import torch
print('torch still CUDA-enabled:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'A dependency replaced torch with a CPU build.'

## 5. Unpack the code and wire up the assets

Weights and video are symlinked rather than copied, so nothing large moves.

In [ ]:
import os, shutil, zipfile

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(f'{DRIVE}/code.zip') as z:
    z.extractall(WORK)

for sub in ('models', 'input_videos', 'output_videos', 'stubs'):
    os.makedirs(f'{WORK}/{sub}', exist_ok=True)

def link(src, dst):
    if not os.path.exists(dst):
        os.symlink(src, dst)

link(f'{DRIVE}/best.pt', f'{WORK}/models/best.pt')
link(f'{DRIVE}/videos/{VIDEO_NAME}', f'{WORK}/input_videos/{VIDEO_NAME}')

os.chdir(WORK)
print(os.getcwd())
print(sorted(os.listdir()))

## 6. Benchmark before committing to a long run

Measures real throughput on this GPU at both resolutions, and counts how
often the ball is found. Local CPU baseline for comparison:
**0.53 s/frame at 640, 2.84 s/frame at 1280.**

In [ ]:
import time
import cv2
from ultralytics import YOLO

cap = cv2.VideoCapture(f'input_videos/{VIDEO_NAME}')
fps = cap.get(cv2.CAP_PROP_FPS)
cap.set(cv2.CAP_PROP_POS_FRAMES, int(float(START) * fps))
frames = []
for _ in range(60):
    ok, f = cap.read()
    if not ok:
        break
    frames.append(f)
cap.release()
print(f'{len(frames)} frames, {frames[0].shape[1]}x{frames[0].shape[0]}')

model = YOLO('models/best.pt')
ball_id = [k for k, v in model.names.items() if v == 'ball'][0]

for size in (640, 1280):
    model.predict(frames[:8], imgsz=size, device=0, half=True, verbose=False)
    torch.cuda.synchronize()
    t0 = time.time()
    out = model.predict(frames, imgsz=size, conf=0.1, device=0, half=True,
                        batch=BATCH, verbose=False)
    torch.cuda.synchronize()
    per = (time.time() - t0) / len(frames)
    hits = sum(1 for r in out if (r.boxes.cls == ball_id).sum() > 0)
    cpu = 0.53 if size == 640 else 2.84
    print(f'imgsz={size}: {per*1000:6.1f} ms/frame  ({cpu/per:5.1f}x the CPU)'
          f'   ball in {hits}/{len(frames)} = {100*hits/len(frames):.0f}%')

## 7. Run the pipeline

Detections are cached in `stubs/` keyed by video, section **and** imgsz, so
re-running only the drawing stages is quick.

A calibration is needed for the section — `calibrations.json` travels in the
zip, so sections already calibrated locally work straight away.

In [ ]:
!python main.py input_videos/{VIDEO_NAME} --start {START} --end {END} \
    --device 0 --half --imgsz {IMGSZ} --batch {BATCH}

## 8. Copy results back to Drive

Colab storage is wiped when the session ends. Stubs are worth keeping —
they are the expensive part.

In [ ]:
import glob, os, shutil

for sub in ('output_videos', 'stubs'):
    dest = f'{DRIVE}/{sub}'
    os.makedirs(dest, exist_ok=True)
    for src in glob.glob(f'{sub}/*'):
        if os.path.isfile(src):
            shutil.copy2(src, dest)
            print('copied', src, '->', dest)

shutil.copy2('calibrations.json', DRIVE)
print('done')